In [1]:
!hdfs dfs -mkdir /analisis_logs
!hdfs dfs -put logfiles.log /analisis_logs
!head -n 100 logfiles.log > test_logfiles.log

## 1. Estadísticas básicas

**Contador de Códigos de Estado HTTP**

Queremos saber cuántas peticiones resultaron exitosas (200), cuántas no encontradas (404), errores de servidor (500), etc.

**Mapper**

In [24]:
%%writefile mapper_1.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    words = line.strip().split()
    print(f"{words[8]}\t1")

Overwriting mapper_1.py


**Reducer**

In [25]:
%%writefile reducer_1.py
#!/usr/bin/env python3
import sys

current_code = 0
current_count = 0
for line in sys.stdin:
    code, count = line.strip().split("\t")
    code = int(code)

    if code == current_code:
        current_count += 1
    else:
        if current_code:
            print(f"{current_code}: {current_count}")
        current_code = code
        current_count = 0

if current_code:
     print(f"{current_code}: {current_count}")

Overwriting reducer_1.py


**Ejecución**

In [26]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_1.py \
-file reducer_1.py \
-mapper mapper_1.py \
-reducer reducer_1.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida

2025-12-16 10:07:15,190 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_1.py, reducer_1.py, /tmp/hadoop-unjar2209863703589220061/] [] /tmp/streamjob5221363839232555080.jar tmpDir=null
2025-12-16 10:07:15,914 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:16,107 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:16,238 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida already exists
Streaming Command Failed!


**Tráfico Total por IP**

En este segundo ejercicio el objetivo será identificar qué direcciones IP están consumiendo más ancho de banda.

**Mapper**


In [5]:
%%writefile mapper_2.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    words = line.strip().split()

    print(f"{words[0]}\t{words[9]}")

Writing mapper_2.py


**Reducer**

In [27]:
%%writefile reducer_2.py
#!/usr/bin/env python3
import sys

current_ip = None
current_byte_count = 0
for line in sys.stdin:
    ip, byte_count = line.strip().split("\t")
    try:
        byte_count = float(byte_count)
    except ValueError:
        byte_count = 0
    if current_ip == ip:
        current_byte_count += byte_count
    else:
        if current_ip:
            print(f"{current_ip}\t{current_byte_count}")
            
        current_ip = ip
        current_byte_count = byte_count

print(f"{current_ip}\t{current_byte_count}")

Overwriting reducer_2.py


**Ejecución**


In [28]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_2.py \
-file reducer_2.py \
-mapper mapper_2.py \
-reducer reducer_2.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida/ip_bytes

2025-12-16 10:07:30,463 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_2.py, reducer_2.py, /tmp/hadoop-unjar4452986992419625473/] [] /tmp/streamjob5729311305938980039.jar tmpDir=null
2025-12-16 10:07:31,163 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:31,340 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:31,445 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida/ip_bytes already exists
Streaming Command Failed!


## 2. Análisis de comportamiento

**URLs más populares**

El objetivo en este ejercicio será encontrar las las rutas (/usr/admin, /usr/register) más solicitadas.

**Mapper**


In [29]:
%%writefile mapper_3.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    words = line.strip().split()
    print(f"{words[5]}{words[6]}{words[7]}\t1")

Overwriting mapper_3.py


**Reducer**

In [30]:
%%writefile reducer_3.py
#!/usr/bin/env python3
import sys

current_url = None
current_count = 0
for line in sys.stdin:
    url, count = line.strip().split("\t")
    if current_url == url:
        current_count += 1
    else:
        if current_url:
            print(f"{current_url}: {current_count}")
        
        current_url = url
        current_count = 0

print(f"{current_url}: {current_count}")

Overwriting reducer_3.py


**Ejecución**


In [31]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_3.py \
-file reducer_3.py \
-mapper mapper_3.py \
-reducer reducer_3.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida/urls

2025-12-16 10:07:45,034 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_3.py, reducer_3.py, /tmp/hadoop-unjar3967188579111058759/] [] /tmp/streamjob6625668285614736037.jar tmpDir=null
2025-12-16 10:07:45,987 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:46,162 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:46,304 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida/urls already exists
Streaming Command Failed!


**Distribución por Método HTTP**

Aquí queremos saber qué tipo de acciones hacen los usuarios (GET vs POST vs DELETE).

**Mapper**


In [32]:
%%writefile mapper_4.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    words = line.strip().split()
    print(f"{words[5]}\t1")

Overwriting mapper_4.py


**Reducer**

In [33]:
%%writefile reducer_4.py
#!/usr/bin/env python3
import sys

current_http = None
current_count = 0
for line in sys.stdin:
    http, count = line.strip().split("\t")
    if current_http == http:
        current_count += 1
    else:
        if current_http:
            print(f"{current_http}: {current_count}")
        
        current_http = http
        current_count = 0

print(f"{current_http}: {current_count}")

Overwriting reducer_4.py


**Ejecución**


In [34]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_4.py \
-file reducer_4.py \
-mapper mapper_4.py \
-reducer reducer_4.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida/http

2025-12-16 10:07:54,399 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_4.py, reducer_4.py, /tmp/hadoop-unjar6410542474057363345/] [] /tmp/streamjob8852030125864920038.jar tmpDir=null
2025-12-16 10:07:55,240 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:55,453 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:55,591 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida/http already exists
Streaming Command Failed!


**Análisis de navegadores**

El objetivo aquí es saber si los usuarios usan Chrome, Firefox, o si son bots/móviles.



**Mapper**


In [15]:
%%writefile mapper_5.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    words = line.strip().split()
    try:
        browser, port = words[-2].split("/")
        print(f"{browser}\t1")
    except ValueError:
        continue

Writing mapper_5.py


**Reducer**

In [35]:
%%writefile reducer_5.py
#!/usr/bin/env python3
import sys

current_browser = None
current_count = 0
for line in sys.stdin:
    browser, count = line.strip().split("\t")
    if current_browser == browser:
        current_count += 1
    else:
        if current_browser:
            print(f"{current_browser}: {current_count}")
        
        current_browser = browser
        current_count = 0

print(f"{current_browser}: {current_count}")


Overwriting reducer_5.py


**Ejecución**


In [36]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_5.py \
-file reducer_5.py \
-mapper mapper_5.py \
-reducer reducer_5.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida/browser

2025-12-16 10:07:58,493 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_5.py, reducer_5.py, /tmp/hadoop-unjar7851615481025024054/] [] /tmp/streamjob34412480419538842.jar tmpDir=null
2025-12-16 10:07:59,219 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:59,403 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:07:59,501 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida/browser already exists
Streaming Command Failed!


## 3. Análisis temporal y de sesión

**Picos de tráfico por hora**

Queremos descubrir a qué hora del día el servidor recibe más carga.

**Mapper**


In [18]:
%%writefile mapper_6.py
#!/usr/bin/env python3
from datetime import datetime
import sys

for line in sys.stdin:
    words = line.strip().split()
    datetime_string = words[3][1:]
    date_time = datetime.strptime(datetime_string, "%d/%b/%Y:%H:%M:%S")
    print(f"{date_time.hour}\t1")

Writing mapper_6.py


**Reducer**

In [37]:
%%writefile reducer_6.py
#!/usr/bin/env python3
import sys

current_hour = None
current_count = 0
for line in sys.stdin:
    hour, count = line.strip().split("\t")
    if current_hour == hour:
        current_count += 1
    else:
        if current_hour:
            print(f"{current_hour}: {current_count}")
        
        current_hour = hour
        current_count = 0

print(f"{current_hour}: {current_count}")

Overwriting reducer_6.py


**Ejecución**


In [38]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_6.py \
-file reducer_6.py \
-mapper mapper_6.py \
-reducer reducer_6.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida/hour

2025-12-16 10:08:04,416 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_6.py, reducer_6.py, /tmp/hadoop-unjar1967041103224234715/] [] /tmp/streamjob1258856399372631424.jar tmpDir=null
2025-12-16 10:08:05,273 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:08:05,445 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:08:05,567 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida/hour already exists
Streaming Command Failed!


**Tasa de error por endpoint**

En este ejercicio queremos descubrir qué URLs están fallando más.



**Mapper**


In [39]:
%%writefile mapper_7.py
#!/usr/bin/env python3
import sys

for line in sys.stdin:
    words = line.strip().split()
    url, http_code = words[6], int(words[8])
    status = None
    if http_code >= 400:
        status = (0, 1)
    elif http_code < 400:
        status = (1, 0)
    print(f"({url}, {status})")

Overwriting mapper_7.py


**Reducer**

In [40]:
%%writefile reducer_7.py
#!/usr/bin/env python3
import sys

current_url = None
total_success = 0
total_errors = 0
error_percent = 0
for line in sys.stdin:
    try:
        url, status_success, status_error = line.strip("(").split(",")
        status_success = int(status_success[2:])
        status_error = int(status_error[:-3])

        if current_url == url:
            if status_success == 1:
                total_success += 1
    
            if status_error == 1:
                total_errors += 1
        else:
            if current_url:
                error_percent = (total_errors / (total_errors + total_success)) * 100
                print(f"{current_url}: {error_percent}%")
            
            current_url = url
            total_success = 0
            total_errors = 0
            error_percent = 0
    except ValueError:
        print("Fallo", status_success, status_error)

print(f"{current_url}: {error_percent}%")

Overwriting reducer_7.py


**Ejecución**


In [41]:
!hadoop jar \
/usr/local/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.0.jar \
-file mapper_7.py \
-file reducer_7.py \
-mapper mapper_7.py \
-reducer reducer_7.py \
-input /analisis_logs/logfiles.log \
-output /analisis_logs/salida/error_percent

2025-12-16 10:08:12,136 WARN streaming.StreamJob: -file option is deprecated, please use generic option -files instead.
packageJobJar: [mapper_7.py, reducer_7.py, /tmp/hadoop-unjar5676472889551861084/] [] /tmp/streamjob8842768685210851662.jar tmpDir=null
2025-12-16 10:08:13,175 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:08:13,605 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at yarnmanager/172.18.0.4:8032
2025-12-16 10:08:13,879 ERROR streaming.StreamJob: Error Launching job : Output directory hdfs://namenode:9000/analisis_logs/salida/error_percent already exists
Streaming Command Failed!
